In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")   
)

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. **Install Ollama** by visiting the download page and following the instructions for your operating system:
	* For **macOS**, download the `.pkg` and install it.
	* For **Windows**, download the `.msi` and install it.
	* For **Linux**, run the following command in the terminal: `curl -fsSL https://ollama.com/install.sh | sh`.
2. **Start the Ollama server** by running the following command in a terminal: `ollama run llama3`. This will:
	* Download the LLaMA 3 model (~4GB).
	* Start the model locally.
	* Open a chat-like interface where you can type questions.
3. **Test the Ollama local server** by running the following command in a terminal: `curl http://localhost:11434`. You should receive a response similar to: `{"models": [...]}`
4. **Install the Python client** with the following command: `pip install ollama`.
5. **Use the Python client** in your code by importing the `ollama` module and using the `ollama.chat()` function to interact with the Ollama server.

In [5]:
answer = assistant.rag("How do I run Olamma locally?")
print(answer)

Based on the provided text, it appears that you are trying to run Olamma locally. However, since Olamma is not explicitly mentioned in the text, I'll make an educated guess that you want to know how to run it locally, especially since Module 1: RAG mentions setting up Python, `uv`, and Jupyter among other tools.

To run a RAG (Relational Augmented Generation) model like Olamma locally, you will need to:

1. **Install Python and Docker**: Olamma is based on Python, so you need to install Python on your system. Then, install Docker, which is used to create and manage containers for your Python environment.
2. **Create a virtual environment**: Use a tool like `virtualenv` or `uv` to create a virtual environment for your Olamma project. This ensures that the Python dependencies required by Olamma are installed and managed within the virtual environment, without affecting your system Python.
3. **Set up Jupyter**: Olamma might use Jupyter for experimentation or visualization. You can set up

In [6]:
# Asking without tools
messages = [
    {"role": "system", "content": "You must use the provided 'search' tool when answering questions about the course using valid tool calls."},
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='llama-3.1-8b-instant',
    input=messages,
)

response.output_text


"I can't take the course with you, but I can provide information and tools to help you learn more about the subject matter. What would you like to know about the course?"

In [7]:
# Defining the tool
def search(query):
    boost_dict={"question":3.0, "section":0.5}
    filter_dict = {"course":"llm-zoomcamp"}

    return index.search(
        query,
        num_results = 5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",  # name alanı burada, function alt nesnesinin içinde olmalı!
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [9]:
response = openai_client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=messages,
    tools=[search_tool],
    tool_choice="auto"
)

In [10]:
import json

response_message = response.choices[0].message

if response_message.tool_calls:
    tool_call = response_message.tool_calls[0]
    function_name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    print(f"Model function called: {function_name}")
    print(f"Arguments: {args}")
        
    results = search(**args)
    result_json = json.dumps(results, indent=2)

    messages.append(response_message)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result_json
    })

    second_response = openai_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=messages,
        tools=[search_tool],
        tool_choice="auto"
    )
    print("\nFinal Answer:")
    print(second_response.choices[0].message.content)

else:
    print("Model did not call a tool, direct response:")
    print(response_message.content)

Model did not call a tool, direct response:
No information is available about the current status of course enrollment.


### 14. Agentic Loop

In [11]:
import json

def make_call(tool_call):
    args = json.loads(tool_call.function.arguments)

    if tool_call.function.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent = 2)

    return{
        "role" : "tool",
        "tool_call_id" : tool_call.id,
        "content" : result_json
    }

In [12]:
def agent_loop(instructions, question, model="llama-3.1-8b-instant") -> str:
    messages = [
        {"role" : "system", "content" : instructions},
        {"role": "user", "content" : question}
    ]

    it = 1

    while True:
        print(f"Iteration #{it}...")
        has_function_calls = False

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool],
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        if response_message.tool_calls:
            has_function_calls = True
            for tool_call in response_message.tool_calls:
                print("function_call:", tool_call.function.name, tool_call.function.arguments)
                
                # Fonksiyonu çalıştırıp çıktıyı alıyoruz ve geçmişe ekliyoruz
                tool_output_msg = make_call(tool_call)
                messages.append(tool_output_msg)
        else:
            # Model tool çağırmadıysa demek ki final cevabını verdi
            last_answer = response_message.content
            print("ASSISTANT:")
            print(last_answer)

        it = it + 1
        if not has_function_calls:
            break
        
    return last_answer

#### TESTS

In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

If the user query contains a potential typo, correct it before searching.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

CRITICAL: At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "How do I run Olamma locally?");

Iteration #1...
function_call: search {"query":"Olamma locally run tutorial"}
function_call: search {"query":"Olamma locally setup"}
Iteration #2...
ASSISTANT:
It appears that the course provides instructions on how to run Olamma locally in one of the answers.

According to the answer "Do I have to use OpenAI, or can I use a different provider?" you can use a different provider and the course code works with only a `base_url` change. 

You can serve a model locally with Ollama, vLLM, LM Studio, or anything else.

So, to run Olamma locally, you can use Ollama. 

Here is the instruction from the answer: 
"Serve a model locally with [Ollama](https://ollama.com/), [vLLM](https://github.com/vllm-project/vllm), LM Studio, or anything else — no external API call at all, so regional blocks don't apply and you don't need a paid key. Most of these also expose an OpenAI-compatible endpoint, so the course code works with only a `base_url` change."

Please note that you may need to adjust the cours

In [18]:
strict_instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(strict_instructions, "what's queen gambit?");

Iteration #1...
function_call: search {"query":"queen gambit course"}
function_call: search {"query":"gambit course"}
function_call: search {"query":"queen gambit meaning"}
Iteration #2...
ASSISTANT:
The question "what's queen gambit" seems off-topic, based on the returned search results. The search results don't mention anything about the Queen's Gambit, which is likely a chess term, so it doesn't seem related to this course.

If you have other areas in the course or its logistics you'd like to explore, I'd be happy to try and help!
